# Anatomy of a Gemma 4 Coding-Agent Submission

A submission is a small declarative package: one root `agent.yaml` connects a prompt and sampling settings, while `eval_config.yaml` sets operational limits. This notebook builds that package, checks its archive and YAML includes locally, and follows a **deterministic mock** coding loop on a tiny toy repository.

Figures report settings read from captured public notebook sources or counts produced by the toy test run. The mock is not Gemma behavior, a benchmark result, or a score. This notebook makes no score claims.

**Run constraints:** CPU only, no internet. Input lookup checks `/kaggle/input/competitions/gemma-4-developer-agent` first, then `/kaggle/input/gemma-4-developer-agent`; the package build does not depend on either path.

In [1]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile, ZipInfo, ZIP_DEFLATED
import hashlib, json, re, subprocess, sys, tempfile, time
import matplotlib.pyplot as plt
import yaml
from IPython.display import Markdown, display

TITLE = "Anatomy of a Gemma 4 Coding Agent Submission"
KERNEL_ID = "prvsiyan/anatomy-of-a-gemma-4-coding-agent-submission"
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_PATH = OUTPUT_ROOT / "submission.zip"
FIGURE_DIR = OUTPUT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
input_candidates = [Path("/kaggle/input/competitions/gemma-4-developer-agent"), Path("/kaggle/input/gemma-4-developer-agent")]
competition_input = next((p for p in input_candidates if p.is_dir()), None)
print("COMPETITION_INPUT=" + (str(competition_input) if competition_input else "not mounted; not required"))
print("RUNTIME=" + json.dumps({"python": sys.version.split()[0], "platform": sys.platform, "matplotlib": plt.matplotlib.__version__}, sort_keys=True))

## 1. The package contract

The captured harness describes `agent.yaml` as the required root configuration. Its `!include` references resolve against files inside the same archive. `eval_config.yaml` is optional; this example includes its four recognized controls: per-command timeout, tool-call cap, task-time cap, and turn cap.

The example uses the competition's 31B model name from the supplied sample package. The prompt and settings are newly written for this lesson; public notebook source code is not copied.

In [2]:
def lines(*items):
    return "\n".join(items) + "\n"

PACKAGE_FILES = {
    "agent.yaml": lines(
        "name: anatomy_agent", "model: gemma-4-31b-it-qat-w4a16-ct",
        "instruction: !include prompts/system.md", "tools:",
        "  - run_command", "  - read_file", "  - edit_file", "  - write_file", "  - get_status", "  - submit_patch",
        "generate_content_config: !include configs/sampling.yaml",
    ),
    "configs/sampling.yaml": lines(
        "temperature: 0.2", "top_p: 0.95", "max_output_tokens: 8192", "thinking_config:",
        "  thinking_budget: 4096", "  include_thoughts: true",
    ),
    "eval_config.yaml": lines(
        "evaluation:", "  timeout_seconds: 240", "  max_tool_calls: 28", "  max_time_minutes: 8", "  max_turns: 80",
    ),
    "prompts/system.md": lines(
        "You are a coding agent working in the task repository.",
        "Read the issue, locate the narrowest relevant implementation and test, and make a focused source change.",
        "Do not edit tests. Run a relevant test after the final source edit, inspect the diff, and submit only after checking the result.",
        "Stay within the declared tool and time budgets.",
    ),
}
with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED) as archive:
    for member, content in PACKAGE_FILES.items():
        info = ZipInfo(member, date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = ZIP_DEFLATED
        info.external_attr = 0o100644 << 16
        archive.writestr(info, content.encode("utf-8"))
print("WROTE_ZIP=" + json.dumps({"path": str(ZIP_PATH), "bytes": ZIP_PATH.stat().st_size, "sha256": hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()}, sort_keys=True))

## 2. Local schema and format check

This checker reads the ZIP back, verifies the root-file and member contract, resolves each `!include` only from the archive, and checks fields and bounds described in the captured harness README. It is a local structural check; it does not invoke Kaggle's scorer or official runtime validator.

In [3]:
class PackageLoader(yaml.SafeLoader):
    pass

def include_package_member(loader, node):
    member = loader.construct_scalar(node)
    if member not in PACKAGE_FILES:
        raise ValueError("!include target is not a package member: " + member)
    return PACKAGE_FILES[member] if member.endswith(".md") else yaml.safe_load(PACKAGE_FILES[member])

PackageLoader.add_constructor("!include", include_package_member)
root_configs = {"agent.yaml", "agent.yml", "root_agent.yaml", "root_agent.yml"}
with ZipFile(ZIP_PATH) as archive:
    members = archive.namelist()
    assert archive.testzip() is None, "ZIP CRC check failed"
    assert len(members) == len(set(members)), "duplicate ZIP members"
    assert all(not PurePosixPath(name).is_absolute() and ".." not in PurePosixPath(name).parts for name in members)
    assert len(root_configs.intersection(members)) == 1, "expected one root agent config"
    assert set(members) == set(PACKAGE_FILES), members
    archive_files = {name: archive.read(name).decode("utf-8") for name in members}
agent = yaml.load(archive_files["agent.yaml"], Loader=PackageLoader)
sampling = yaml.safe_load(archive_files["configs/sampling.yaml"])
evaluation = yaml.safe_load(archive_files["eval_config.yaml"])["evaluation"]
assert agent["name"] == "anatomy_agent" and agent["model"] == "gemma-4-31b-it-qat-w4a16-ct"
assert agent["instruction"] == archive_files["prompts/system.md"]
assert agent["generate_content_config"] == sampling
assert set(agent["tools"]) == {"run_command", "read_file", "edit_file", "write_file", "get_status", "submit_patch"}
assert set(sampling) == {"temperature", "top_p", "max_output_tokens", "thinking_config"}
assert sampling["temperature"] >= 0 and 0 <= sampling["top_p"] <= 1
assert 1 <= sampling["max_output_tokens"] <= 32768
assert 0 <= sampling["thinking_config"]["thinking_budget"] <= 32768
assert isinstance(sampling["thinking_config"]["include_thoughts"], bool)
assert set(evaluation) == {"timeout_seconds", "max_tool_calls", "max_time_minutes", "max_turns"}
assert all(type(value) is int and value > 0 for value in evaluation.values())
ZIP_EVIDENCE = {"status": "passed", "checker": "local ZIP/YAML/include/harness-format checks; no official runtime validator", "root_agent_config_count": 1, "members": members, "sampling": sampling, "evaluation": evaluation, "bytes": ZIP_PATH.stat().st_size, "sha256": hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()}
print("FORMAT_CHECK=" + json.dumps(ZIP_EVIDENCE, sort_keys=True))

## 3. What public notebook snapshots configure

These are declared settings transcribed from captured notebook sources, not measurements of consumed budget. Hitarth Jain's row is a reference configuration discussed in a guide, not a claim that the guide itself is the submitted package. Links point to the authors' notebooks.

The local Draft-F attribution records Flexonafft V10 as Apache-2.0. The pulled Kaggle metadata for that notebook has a null license field; the table keeps both facts visible. Captured metadata for the other listed notebooks also has no declared license. This notebook links and analyzes settings only; it does not reproduce their code.

In [4]:
PUBLIC_CONFIGS = [
    {"label":"Flexonafft V10", "kind":"public package snapshot", "link":"https://www.kaggle.com/code/flexonafft/gemma-4-budget-aware-coding-agent-v2", "temperature":0.2, "thinking_budget":4096, "max_tool_calls":28, "max_turns":80, "max_time_minutes":8, "timeout_seconds":240, "license":"Apache-2.0 per lane Draft-F attribution; captured Kaggle metadata: undeclared"},
    {"label":"Honghanhhh base config", "kind":"public package snapshot", "link":"https://www.kaggle.com/code/honghanhhh/gemma-4-baseline-lb-0-24", "temperature":0.2, "thinking_budget":4096, "max_tool_calls":28, "max_turns":80, "max_time_minutes":8, "timeout_seconds":240, "license":"Not declared in captured Kaggle metadata"},
    {"label":"Hitarth Jain reference", "kind":"guide reference config", "link":"https://www.kaggle.com/code/hitarthjain0/gemma-4-agent-what-took-lb-0-12-0-24", "temperature":0.2, "thinking_budget":4096, "max_tool_calls":28, "max_turns":80, "max_time_minutes":8, "timeout_seconds":240, "license":"Not declared in captured Kaggle metadata"},
    {"label":"Cosmorin V24/E07", "kind":"public package snapshot", "link":"https://www.kaggle.com/code/cosmorin/gemma-4-idris-v24-e07-faithful-replica", "temperature":1.0, "thinking_budget":4096, "max_tool_calls":28, "max_turns":80, "max_time_minutes":8, "timeout_seconds":240, "license":"Not declared in captured Kaggle metadata"},
    {"label":"Jingshuifu package", "kind":"public package snapshot", "link":"https://www.kaggle.com/code/jingshuifu/gemma-4-dev-agent-lb024", "temperature":0.2, "thinking_budget":4096, "max_tool_calls":28, "max_turns":80, "max_time_minutes":8, "timeout_seconds":240, "license":"Not declared in captured Kaggle metadata"},
]
header = "| Notebook / source | Type | Calls | Turns | Minutes | Timeout (s) | Temp. | Thinking budget | License note |\n|---|---|---:|---:|---:|---:|---:|---:|---|"
rows = [header]
for row in PUBLIC_CONFIGS:
    rows.append(f"| [{row['label']}]({row['link']}) | {row['kind']} | {row['max_tool_calls']} | {row['max_turns']} | {row['max_time_minutes']} | {row['timeout_seconds']} | {row['temperature']} | {row['thinking_budget']} | {row['license']} |")
display(Markdown("\n".join(rows)))

## 4. Figures: package, budgets, and sampling settings

Each plotted value is either a setting in the linked snapshots or a value in this notebook's example ZIP. The diagrams explain configuration structure; they do not predict benchmark outcomes.

In [5]:
INK, BLUE, TEAL, GOLD, CORAL, PAPER, GRID = "#20282d", "#3178a8", "#4f927c", "#c38b35", "#ba6758", "#fbfaf7", "#e3e0d8"
plt.rcParams.update({"figure.facecolor":PAPER, "axes.facecolor":PAPER, "text.color":INK, "axes.labelcolor":INK, "xtick.color":INK, "ytick.color":INK, "font.size":9})

def card(ax, x, y, w, h, title, body, color):
    ax.add_patch(plt.Rectangle((x,y),w,h,facecolor="white",edgecolor=color,linewidth=1.7))
    ax.text(x+.17,y+h-.25,title,fontsize=10,weight="bold",va="top",color=color)
    ax.text(x+.17,y+h-.58,body,fontsize=8.3,va="top",linespacing=1.3)

# Figure 1: ZIP contents and include links.
fig, ax = plt.subplots(figsize=(10,4.6)); ax.set_xlim(0,10); ax.set_ylim(0,5); ax.axis("off")
ax.text(.35,4.55,"One ZIP, one root agent config",fontsize=15,weight="bold")
ax.text(.35,4.15,"Declarative files are discovered inside the archive",fontsize=9,color="#596268")
card(ax,.35,1.35,3.1,2.25,"submission.zip","agent.yaml  (required root)\nconfigs/sampling.yaml\neval_config.yaml\nprompts/system.md",BLUE)
card(ax,5.25,3.1,4.3,.95,"agent.yaml","Agent + tools; includes prompt and sampling config",TEAL)
card(ax,5.25,1.65,4.3,.95,"eval_config.yaml","Optional task and command budgets",GOLD)
card(ax,5.25,.2,4.3,.95,"Included files","sampling.yaml controls generation; system.md supplies instructions",CORAL)
for y0,y1 in [(2.85,3.5),(2.48,2.12),(2.08,.68)]: ax.annotate("",xy=(5.1,y1),xytext=(3.52,y0),arrowprops={"arrowstyle":"->","color":"#68747a","lw":1.3})
ax.text(3.63,3.37,"contains",fontsize=8,color="#596268"); ax.text(3.63,2.1,"optional",fontsize=8,color="#596268"); ax.text(3.63,.7,"!include",fontsize=8,color="#596268")
fig.tight_layout(); fig.savefig(FIGURE_DIR/"figure_01_package_anatomy.png",dpi=160,bbox_inches="tight"); plt.show(); plt.close(fig)

# Figure 2: declared operational caps in captured sources.
fig, axes = plt.subplots(2,2,figsize=(11,6.8))
metrics=[("max_tool_calls","Tool-call cap","calls",BLUE),("max_turns","Turn cap","turns",TEAL),("max_time_minutes","Task-time cap","minutes",GOLD),("timeout_seconds","Per-command timeout","seconds",CORAL)]
labels=[row["label"] for row in PUBLIC_CONFIGS]
for ax,(key,title,unit,color) in zip(axes.flat,metrics):
    vals=[row[key] for row in PUBLIC_CONFIGS]; ys=list(range(len(labels)))
    ax.barh(ys,vals,color=color,alpha=.84); ax.set_yticks(ys,labels if ax in (axes[0,0],axes[1,0]) else [""]*len(labels)); ax.invert_yaxis()
    ax.set_title(title,loc="left",weight="bold"); ax.set_xlabel(unit); ax.grid(axis="x",color=GRID,linewidth=.7); ax.set_axisbelow(True); ax.spines[["top","right","left"]].set_visible(False)
    for yy,value in zip(ys,vals): ax.text(value+max(vals)*.018,yy,str(value),va="center",fontsize=8)
fig.suptitle("Declared operational budgets in captured public sources",x=.08,ha="left",fontsize=14,weight="bold")
fig.text(.08,.01,"Configuration caps, not observed budget consumption or performance.",fontsize=8,color="#596268")
fig.tight_layout(rect=(0,.04,1,.94)); fig.savefig(FIGURE_DIR/"figure_02_public_budget_settings.png",dpi=160,bbox_inches="tight"); plt.show(); plt.close(fig)

# Figure 3: original numeric values for sampling choices.
fig,axes=plt.subplots(1,2,figsize=(11,4.8)); ys=list(range(len(PUBLIC_CONFIGS))); colors=[BLUE,TEAL,GOLD,CORAL,"#786c9c"]
for ax,key,title,xlabel in [(axes[0],"temperature","Temperature","temperature"),(axes[1],"thinking_budget","Thinking-token budget","tokens")]:
    vals=[row[key] for row in PUBLIC_CONFIGS]; ax.scatter(vals,ys,s=90,c=colors,edgecolors="white",linewidths=.8,zorder=3)
    ax.set_yticks(ys,[row["label"] for row in PUBLIC_CONFIGS]); ax.invert_yaxis(); ax.set_title(title,loc="left",weight="bold"); ax.set_xlabel(xlabel); ax.grid(axis="x",color=GRID,linewidth=.8); ax.set_axisbelow(True); ax.spines[["top","right","left"]].set_visible(False)
    for yy,value in zip(ys,vals): ax.text(value,yy-.18,str(value),ha="center",va="bottom",fontsize=8)
fig.suptitle("Sampling choices in public notebook snapshots",x=.08,ha="left",fontsize=14,weight="bold")
fig.text(.08,.01,"Declared settings only; no score or causal effect is inferred.",fontsize=8,color="#596268")
fig.tight_layout(rect=(0,.05,1,.91)); fig.savefig(FIGURE_DIR/"figure_03_sampling_settings.png",dpi=160,bbox_inches="tight"); plt.show(); plt.close(fig)
print("FIGURES_01_03=" + json.dumps(sorted(p.name for p in FIGURE_DIR.glob("figure_0[1-3]_*.png"))))

## 5. How to read the limits

The four controls have different units and scopes. `max_time_minutes` is a task-level cap; `timeout_seconds` limits one command. Tool calls and turns are separate counters. The diagram uses this notebook's example archive values and labels them as configured limits, not elapsed time.

In [6]:
# Figure 4: scopes and units for the example archive.
fig,ax=plt.subplots(figsize=(10,4)); ax.set_xlim(0,10); ax.set_ylim(0,4.2); ax.axis("off")
ax.text(.3,3.85,"Four caps, four different questions",fontsize=14,weight="bold")
ax.add_patch(plt.Rectangle((.45,2.1),9.1,1.15,facecolor="#eaf2f6",edgecolor=BLUE,linewidth=1.6))
ax.text(.7,2.85,"Task-level wall-clock budget",fontsize=10,weight="bold",color=BLUE); ax.text(.7,2.4,f"max_time_minutes = {evaluation['max_time_minutes']} minutes",fontsize=10)
for x,w,title,body,color in [(1,.3+3.35,"Single command",f"timeout_seconds = {evaluation['timeout_seconds']} s",GOLD),(5.15,1.8,"Tool calls",str(evaluation["max_tool_calls"]),TEAL),(7.3,1.8,"Reasoning turns",str(evaluation["max_turns"]),CORAL)]:
    ax.add_patch(plt.Rectangle((x,.55),w,1,facecolor="white",edgecolor=color,linewidth=1.4)); ax.text(x+.16,1.2,title,fontsize=9,weight="bold",color=color); ax.text(x+.16,.82,body,fontsize=9)
ax.annotate("",xy=(2.75,1.7),xytext=(2.75,2.0),arrowprops={"arrowstyle":"->","color":"#68747a"})
ax.text(.48,.12,"Configured maxima do not mean every run uses every call, turn, or second.",fontsize=8,color="#596268")
fig.tight_layout(); fig.savefig(FIGURE_DIR/"figure_04_budget_scope.png",dpi=160,bbox_inches="tight"); plt.show(); plt.close(fig)

## 6. Deterministic mock loop: fail a test, inspect, retry

The toy repository has three unit tests. A fixed first patch mishandles the boundary case; the mock loop reads the failure and applies a fixed correction. It makes zero model calls and runs only this tiny local test file. Its pass counts describe the fixture, not competition performance.

In [7]:
toy_tests = "\n".join([
    "import unittest", "from access import access_label", "",
    "class AccessLabelTests(unittest.TestCase):",
    "    def test_child(self): self.assertEqual(access_label(12), 'minor')",
    "    def test_boundary(self): self.assertEqual(access_label(18), 'adult')",
    "    def test_adult(self): self.assertEqual(access_label(44), 'adult')",
    "", "if __name__ == '__main__': unittest.main()", "",
])
first_patch = "\n".join(["def access_label(age):", "    return 'adult' if age > 18 else 'minor'", ""])
retry_patch = "\n".join(["def access_label(age):", "    return 'adult' if age >= 18 else 'minor'", ""])
mock_attempts=[]; mock_started=time.perf_counter()
with tempfile.TemporaryDirectory(prefix="gemma_submission_toy_repo_") as temp_dir:
    toy_repo=Path(temp_dir); (toy_repo/"test_access.py").write_text(toy_tests)
    for label,source in [("first attempt",first_patch),("retry after failed test",retry_patch)]:
        (toy_repo/"access.py").write_text(source)
        result=subprocess.run([sys.executable,"-m","unittest","-q","test_access"],cwd=toy_repo,text=True,capture_output=True,timeout=20)
        transcript=result.stdout+result.stderr; ran=re.search(r"Ran (\d+) tests?",transcript)
        failures=re.search(r"failures=(\d+)",transcript); errors=re.search(r"errors=(\d+)",transcript)
        total=int(ran.group(1)) if ran else 0; failed=int(failures.group(1)) if failures else 0; errored=int(errors.group(1)) if errors else 0
        mock_attempts.append({"label":label,"returncode":result.returncode,"tests":total,"passed":total-failed-errored,"failed":failed,"errors":errored})
        if result.returncode==0: break
        assert label=="first attempt" and failed>0, "retry must follow a test failure"
mock_runtime_seconds=time.perf_counter()-mock_started
assert [r["passed"] for r in mock_attempts]==[2,3] and [r["failed"] for r in mock_attempts]==[1,0]
assert mock_attempts[-1]["returncode"]==0
MOCK_EVIDENCE={"mode":"deterministic mock; zero model calls","repo":"temporary toy repository","attempts":mock_attempts,"runtime_seconds":round(mock_runtime_seconds,4),"official_score_claim":False}
print("MOCK_LOOP="+json.dumps(MOCK_EVIDENCE,sort_keys=True))

# Figure 5: observed toy-test counts by attempt.
fig,ax=plt.subplots(figsize=(8.5,4.2)); xs=list(range(len(mock_attempts))); passed=[r["passed"] for r in mock_attempts]; failed=[r["failed"]+r["errors"] for r in mock_attempts]
ax.bar(xs,passed,color=TEAL,label="passed tests"); ax.bar(xs,failed,bottom=passed,color=CORAL,label="failed or errored tests")
ax.set_xticks(xs,[r["label"] for r in mock_attempts]); ax.set_ylabel("Toy unit tests"); ax.set_ylim(0,3.8); ax.set_title("Mock loop: a failed boundary test triggers a retry",loc="left",weight="bold")
ax.grid(axis="y",color=GRID,linewidth=.8); ax.set_axisbelow(True); ax.spines[["top","right"]].set_visible(False)
for x,ok,bad in zip(xs,passed,failed): ax.text(x,ok+bad+.08,f"{ok}/{ok+bad} pass",ha="center",fontsize=9)
ax.legend(frameon=False,loc="upper right"); fig.text(.13,.01,"Toy tests only; not a Gemma run, score, or prediction.",fontsize=8,color="#596268")
fig.tight_layout(rect=(0,.05,1,1)); fig.savefig(FIGURE_DIR/"figure_05_mock_retry.png",dpi=160,bbox_inches="tight"); plt.show(); plt.close(fig)

## Credits, sources, and limits

- [Flexonafft, *Gemma 4 | Budget-Aware Coding Agent v2*](https://www.kaggle.com/code/flexonafft/gemma-4-budget-aware-coding-agent-v2): V10 is credited as Apache-2.0 in the lane's Draft-F attribution record. The captured Kaggle metadata has no declared license; this notebook does not copy its code.
- [Honghanhhh, *Gemma 4 Baseline | LB 0.24*](https://www.kaggle.com/code/honghanhhh/gemma-4-baseline-lb-0-24), [Hitarth Jain, *What Took LB 0.12 → 0.24*](https://www.kaggle.com/code/hitarthjain0/gemma-4-agent-what-took-lb-0-12-0-24), [Cosmorin, *Gemma 4 Idris v24 E07 Faithful Replica*](https://www.kaggle.com/code/cosmorin/gemma-4-idris-v24-e07-faithful-replica), and [Jingshuifu, *Gemma 4 Dev Agent lb024*](https://www.kaggle.com/code/jingshuifu/gemma-4-dev-agent-lb024): author and settings credit; captured metadata does not declare a license for these notebooks.
- Package shape and limits come from the supplied `competition_files/HARNESS_README.md` and sample `agent.yaml`; see the [official competition overview](https://www.kaggle.com/competitions/gemma-4-developer-agent/overview).

The notebook does not run Gemma, use internet, submit an entry, or publish a kernel. Public settings are configuration facts; toy test counts are local fixture results.